In [ ]:
from nn import *
from ray import train, tune


def Ret(i, k, v, d, t, o, h=1, s=1, r=3, bias=False):
    return Add(
        Sequential(
            Add(
                Sequential(
                    Dot(
                        Sequential(Linear(i, h * k), Unflatten(-1, [h, 1, k])),
                        Sequential(
                            Mul(
                                Sequential(Linear(i, h * k), Unflatten(-1, [h, 1, k])),
                                Sequential(Linear(i, h * v, False), Unflatten(-1, [h, v, 1])),
                            ),
                            Decay(k, v, h, s),
                        ),
                    ),
                    Flatten(-3),
                    Linear(h * v, t, False),
                ),
                SepConvLeft(i, d, t, r),
            ),
            BatchNorm(t),
            SiLU(True),
            Linear(t, o, False),
        ),
        Linear(i, o, bias),
    )


def Model(e, i, r=[]):
    return Sequential(
        Embedding(e, i),
        *[Ret(**r, bias=not nbias) for nbias, r in enumerate(r, 1 - len(r))],
    )


def train(
    _,
    a=("py.txt", 32, 256),
    m=("cuda", torch.bfloat16),
    O={"max_lr": 0.005, "total_steps": 10000, "pct_start": 0.01, "div_factor": 9, "final_div_factor": 99},
    compile=torch.compile,
    tqdm=lambda x: x,
    report=train.report,
    sz=100000,
):
    a = Dat(*a)
    _ = param(_)
    m = compile(sized(lambda x: Model(**arg(**_ * x)), sz)).to(*m)
    o = torch.optim.Adam(m.parameters(), eps=5e-41)
    O = torch.optim.lr_scheduler.OneCycleLR(o, **O)
    X = a()
    L = A = 0
    for _ in tqdm(range(1, O.total_steps + 1)):
        x, X = X, a()
        y = m(x[:, :-1])
        x = x[:, 1:]
        l = functional.cross_entropy(y.mT, x)
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        L += l.detach()
        A += (y.argmax(2) == x).sum()
        if not _ % 100:
            sz = size_select(m)
            report({"l": float(L), "a": float(A) / x.numel()})
            L = A = 0

In [ ]:
tuner = tune.Tuner(
    tune.with_resources(train, {"gpu": 0.33}),
    param_space=dict(
        e=68,
        i_r0i=1.0,
        r0h=tune.loguniform(0.1, 0.8),
        r0k=tune.loguniform(0.1, 0.8),
        r0v=tune.loguniform(0.1, 0.8),
        r0d=tune.loguniform(0.1, 0.8),
        r0t=tune.loguniform(0.5, 4),
        r0o_r1i=tune.loguniform(0.5, 4),
        r1h=tune.loguniform(0.1, 0.8),
        r1k=tune.loguniform(0.1, 0.8),
        r1v=tune.loguniform(0.1, 0.8),
        r1d=tune.loguniform(0.1, 0.8),
        r1t=tune.loguniform(0.5, 4),
        r1o_r2i=tune.loguniform(0.5, 4),
        r2h=tune.loguniform(0.1, 0.8),
        r2k=tune.loguniform(0.1, 0.8),
        r2v=tune.loguniform(0.1, 0.8),
        r2d=tune.loguniform(0.1, 0.8),
        r2t=tune.loguniform(0.5, 4),
        r2o_r3i=tune.loguniform(0.5, 4),
        r3h=tune.loguniform(0.1, 0.8),
        r3k=tune.loguniform(0.1, 0.8),
        r3v=tune.loguniform(0.1, 0.8),
        r3d=tune.loguniform(0.1, 0.8),
        r3t=tune.loguniform(0.5, 4),
        r3o=68,
        r0s=tune.randint(0, 3 + 1),
        r0r=tune.randint(1, 5 + 1),
        r1s=tune.randint(0, 3 + 1),
        r1r=tune.randint(1, 5 + 1),
        r2s=tune.randint(0, 3 + 1),
        r2r=tune.randint(1, 5 + 1),
        r3s=tune.randint(0, 3 + 1),
        r3r=tune.randint(1, 5 + 1),
    ),
    tune_config=tune.TuneConfig(
        mode="min",
        metric="l",
        num_samples=500,
        max_concurrent_trials=3,
        scheduler=tune.schedulers.ASHAScheduler(max_t=100, grace_period=3),
    ),
)
a10 = tuner.fit()

In [ ]:
df = a10.get_dataframe()
df = df[df.training_iteration == 100][["l", "a"] + [i for i in df.columns if "/" in i]].rename(columns=lambda x: x.split("/")[-1])[:10]
df.agg("min mean median max std".split()).T

In [ ]:
tuner = tune.Tuner(
    tune.with_resources(
        tune.with_parameters(
            train,
            sz=100000,
            O={"max_lr": 0.002, "total_steps": 10000, "pct_start": 0.01, "div_factor": 9, "final_div_factor": 99},
        ),
        {"gpu": 0.5},
    ),
    param_space=dict(
        e=68,
        i_r0i=1.0,
        r0h=tune.loguniform(0.1, 0.8),
        r0k=tune.loguniform(0.1, 0.8),
        r0v=tune.loguniform(0.1, 0.8),
        r0d=tune.loguniform(0.1, 0.8),
        r0t=tune.loguniform(0.5, 4),
        r0o_r1i=tune.loguniform(0.5, 4),
        r1h=tune.loguniform(0.1, 0.8),
        r1k=tune.loguniform(0.1, 0.8),
        r1v=tune.loguniform(0.1, 0.8),
        r1d=tune.loguniform(0.1, 0.8),
        r1t=tune.loguniform(0.5, 4),
        r1o_r2i=tune.loguniform(0.5, 4),
        r2h=tune.loguniform(0.1, 0.8),
        r2k=tune.loguniform(0.1, 0.8),
        r2v=tune.loguniform(0.1, 0.8),
        r2d=tune.loguniform(0.1, 0.8),
        r2t=tune.loguniform(0.5, 4),
        r2o_r3i=tune.loguniform(0.5, 4),
        r3h=tune.loguniform(0.1, 0.8),
        r3k=tune.loguniform(0.1, 0.8),
        r3v=tune.loguniform(0.1, 0.8),
        r3d=tune.loguniform(0.1, 0.8),
        r3t=tune.loguniform(0.5, 4),
        r3o=68,
        r0s=tune.randint(0, 3 + 1),
        r0r=tune.randint(1, 5 + 1),
        r1s=tune.randint(0, 3 + 1),
        r1r=tune.randint(1, 5 + 1),
        r2s=tune.randint(0, 3 + 1),
        r2r=tune.randint(1, 5 + 1),
        r3s=tune.randint(0, 3 + 1),
        r3r=tune.randint(1, 5 + 1),
    ),
    tune_config=tune.TuneConfig(
        mode="max",
        metric="a",
        num_samples=500,
        max_concurrent_trials=2,
        scheduler=tune.schedulers.ASHAScheduler(max_t=100, grace_period=2),
    ),
)
a11 = tuner.fit()

In [ ]:
df = a11.get_dataframe()
df = df[df.training_iteration == 100][["l", "a"] + [i for i in df.columns if "/" in i]].rename(columns=lambda x: x.split("/")[-1])[:10]
display(df.T)
df.agg("min mean median max std".split()).T

In [ ]:
tuner = tune.Tuner(
    tune.with_resources(
        tune.with_parameters(
            train,
            sz=100000,
            O={"max_lr": 0.002, "total_steps": 10000, "pct_start": 0.01, "div_factor": 9, "final_div_factor": 99},
        ),
        {"gpu": 0.5},
    ),
    param_space=dict(
        e=68,
        i_r0i=1.0,
        r0h=tune.loguniform(0.1, 0.8),
        r0k=tune.loguniform(0.1, 0.8),
        r0v=tune.loguniform(0.1, 0.8),
        r0d=tune.loguniform(0.1, 0.8),
        r0t=tune.loguniform(0.5, 4),
        r0o_r1i=tune.loguniform(0.5, 4),
        r1h=tune.loguniform(0.1, 0.8),
        r1k=tune.loguniform(0.1, 0.8),
        r1v=tune.loguniform(0.1, 0.8),
        r1d=tune.loguniform(0.1, 0.8),
        r1t=tune.loguniform(0.5, 4),
        r1o_r2i=tune.loguniform(0.5, 4),
        r2h=tune.loguniform(0.1, 0.8),
        r2k=tune.loguniform(0.1, 0.8),
        r2v=tune.loguniform(0.1, 0.8),
        r2d=tune.loguniform(0.1, 0.8),
        r2t=tune.loguniform(0.5, 4),
        r2o_r3i=tune.loguniform(0.5, 4),
        r3h=tune.loguniform(0.1, 0.8),
        r3k=tune.loguniform(0.1, 0.8),
        r3v=tune.loguniform(0.1, 0.8),
        r3d=tune.loguniform(0.1, 0.8),
        r3t=tune.loguniform(0.5, 4),
        r3o_r4i=tune.loguniform(0.5, 4),
        r4h=tune.loguniform(0.1, 0.8),
        r4k=tune.loguniform(0.1, 0.8),
        r4v=tune.loguniform(0.1, 0.8),
        r4d=tune.loguniform(0.1, 0.8),
        r4t=tune.loguniform(0.5, 4),
        r4o_r5i=tune.loguniform(0.5, 4),
        r5h=tune.loguniform(0.1, 0.8),
        r5k=tune.loguniform(0.1, 0.8),
        r5v=tune.loguniform(0.1, 0.8),
        r5d=tune.loguniform(0.1, 0.8),
        r5t=tune.loguniform(0.5, 4),
        r5o_r6i=tune.loguniform(0.5, 4),
        r6h=tune.loguniform(0.1, 0.8),
        r6k=tune.loguniform(0.1, 0.8),
        r6v=tune.loguniform(0.1, 0.8),
        r6d=tune.loguniform(0.1, 0.8),
        r6t=tune.loguniform(0.5, 4),
        r6o=68,
        r0s=tune.randint(0, 3 + 1),
        r0r=tune.randint(1, 5 + 1),
        r1s=tune.randint(0, 3 + 1),
        r1r=tune.randint(1, 5 + 1),
        r2s=tune.randint(0, 3 + 1),
        r2r=tune.randint(1, 5 + 1),
        r3s=tune.randint(0, 3 + 1),
        r3r=tune.randint(1, 5 + 1),
        r4s=tune.randint(0, 3 + 1),
        r4r=tune.randint(1, 5 + 1),
        r5s=tune.randint(0, 3 + 1),
        r5r=tune.randint(1, 5 + 1),
        r6s=tune.randint(0, 3 + 1),
        r6r=tune.randint(1, 5 + 1),
    ),
    tune_config=tune.TuneConfig(
        mode="max",
        metric="a",
        num_samples=2000,
        max_concurrent_trials=2,
        scheduler=tune.schedulers.ASHAScheduler(max_t=100, grace_period=2),
    ),
)
a12 = tuner.fit()

In [ ]:
df = a12.get_dataframe()
df = df[df.training_iteration == 100][["l", "a"] + [i for i in df.columns if "/" in i]].rename(columns=lambda x: x.split("/")[-1])[:10]
df.agg("min mean median max std count".split()).T